# **Imports**

In [ ]:
!pip install datasets
!pip install evaluate

In [ ]:
from datasets import DatasetDict, Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification,TrainingArguments, Trainer
import evaluate
import numpy as np
from transformers import DataCollatorWithPadding, EarlyStoppingCallback

# **Load Dataset**

In [ ]:
dataset_dict = load_dataset("google/code_x_glue_cc_code_completion_line", "python")

# **Load Pre-Trained model**

In [ ]:
from transformers import AutoModelForCausalLM

# using GPT-2 Medium for better generative performance
model_path = "gpt2"

# load model tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)
# GPT-2 does not have a native pad token, so we use the EOS token
tokenizer.pad_token = tokenizer.eos_token

# Load GPT-2 Medium with a Causal Language Modeling head
model = AutoModelForCausalLM.from_pretrained(model_path)
model.config.pad_token_id = tokenizer.eos_token_id

# **Set Trainable Parameters**

In [ ]:
# freeze all base model parameters
for param in model.parameters():
    param.requires_grad = True

# Unfreeze the last transformer block and the LM head for fine-tuning
# GPT-2 Medium has 24 layers (0-23)
#for name, param in model.named_parameters():
#    if "transformer.h.23" in name or "lm_head" in name:
#        param.requires_grad = True

# **Data Pre-Processing**

In [ ]:
# define text preprocessing for CodeXGLUE Line-level Completion
def preprocess_function(examples):
    # The dataset 'google/code_x_glue_cc_code_completion_line' uses 'input' as the column name
    inputs = tokenizer(examples["input"], truncation=True, padding="max_length", max_length=512)

    # For Causal LM (GPT-2), the labels are the input_ids
    inputs["labels"] = inputs["input_ids"].copy()
    return inputs

# Get column names to remove them during mapping
column_names = dataset_dict["train"].column_names

# preprocess all datasets and remove old columns
tokenized_data = dataset_dict.map(
    preprocess_function,
    batched=True,
    remove_columns=column_names
)

In [ ]:
# split the train set: 90% for training, 10% for validation
split_data = tokenized_data["train"].train_test_split(test_size=0.1)

# now you have train and test (which we will use as validation)
tokenized_data["train"] = split_data["train"]
tokenized_data["validation"] = split_data["test"]

print(tokenized_data.keys())
# now this will return ['train', 'validation']

In [ ]:
from transformers import DataCollatorForLanguageModeling

# create data collator for language modeling (causal)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# **Example Code: Fine-Tuning GPT-2 for Code Prediction**
# **Define Evaluation Metrics**

In [ ]:
import math

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # shift so that tokens < n predict n
    # this is often handled by the Trainer/Loss, so we compute perplexity from the loss if available
    # or simply return an empty dict if using Trainer's built-in logging
    return {}

# **Training Parameters**

In [ ]:
# hyperparameters for GPT-2 Medium
lr = 2e-4
batch_size = 4 # reduced batch size for GPT-2 Medium to fit in memory
num_epochs = 3

training_args = TrainingArguments(
    output_dir="gpt2-code-completion",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=1,
    num_train_epochs=num_epochs,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    fp16=True,
    report_to="none",
    gradient_checkpointing=True,
    prediction_loss_only=True,

)

### Note on Learning Rate
If the validation loss starts increasing while training loss decreases, you may be overfitting or the learning rate is too high.
- **Current:** `2e-4` (Good for training just the head)
- **Alternative:** `5e-5` or `2e-5` (Better if you decide to unfreeze more layers)

# **Fine-tune Model**

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable: {trainable:,}")
print(f"Total: {total:,}")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"], 
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# Start the training process
trainer.train()

In [ ]:
import os
print(os.listdir("gpt2-code-completion"))

# **Validation Data**

In [4]:
# # apply model to validation dataset
# predictions=trainer.predict(tokenized_data["validation"])

# # extract the logits and labels from the predictions object
# logits=predictions.predictions
# labels=predictions.label_ids

# # use your compute_metrics function
# metrics=compute_metrics((logits, labels))
# print(metrics)

eval_results = trainer.evaluate()

val_loss = eval_results["eval_loss"]
perplexity = math.exp(val_loss)

print("Validation Loss:", val_loss)
print("Perplexity:", perplexity)

In [ ]:
#!nvidia-smi

In [ ]:
from transformers import GPT2LMHeadModel
model = GPT2LMHeadModel.from_pretrained(
    "gpt2-code-completion/checkpoint-846"
)

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained(
    "gpt2-code-completion/checkpoint-846"
)

model = GPT2LMHeadModel.from_pretrained(
    "gpt2-code-completion/checkpoint-846"
)

In [ ]:
import torch

prompt = "import pandas as pd\nimport numpy as np\n\n"

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [1]:
import os
print(os.listdir("gpt2-code-completion/checkpoint-846"))

['generation_config.json', 'trainer_state.json', 'model.safetensors', 'config.json', 'scaler.pt', 'training_args.bin', 'tokenizer.json', 'tokenizer_config.json', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth']


In [2]:
!zip -r checkpoint-846.zip gpt2-code-completion/checkpoint-846

  adding: gpt2-code-completion/checkpoint-846/ (stored 0%)
  adding: gpt2-code-completion/checkpoint-846/generation_config.json (deflated 25%)
  adding: gpt2-code-completion/checkpoint-846/trainer_state.json (deflated 74%)
  adding: gpt2-code-completion/checkpoint-846/model.safetensors (deflated 7%)
  adding: gpt2-code-completion/checkpoint-846/config.json (deflated 53%)
  adding: gpt2-code-completion/checkpoint-846/scaler.pt (deflated 64%)
  adding: gpt2-code-completion/checkpoint-846/training_args.bin (deflated 53%)
  adding: gpt2-code-completion/checkpoint-846/tokenizer.json (deflated 82%)
  adding: gpt2-code-completion/checkpoint-846/tokenizer_config.json (deflated 48%)
  adding: gpt2-code-completion/checkpoint-846/optimizer.pt (deflated 8%)
  adding: gpt2-code-completion/checkpoint-846/scheduler.pt (deflated 61%)
  adding: gpt2-code-completion/checkpoint-846/rng_state.pth (deflated 26%)


In [6]:
!zip model_only.zip \
gpt2-code-completion/checkpoint-846/model.safetensors \
gpt2-code-completion/checkpoint-846/config.json \
gpt2-code-completion/checkpoint-846/tokenizer.json \
gpt2-code-completion/checkpoint-846/tokenizer_config.json \
gpt2-code-completion/checkpoint-846/generation_config.json

updating: gpt2-code-completion/checkpoint-846/model.safetensors (deflated 7%)
updating: gpt2-code-completion/checkpoint-846/config.json (deflated 53%)
updating: gpt2-code-completion/checkpoint-846/tokenizer.json (deflated 82%)
updating: gpt2-code-completion/checkpoint-846/tokenizer_config.json (deflated 48%)
  adding: gpt2-code-completion/checkpoint-846/generation_config.json (deflated 25%)
